In [25]:
import numpy as np
import pandas as pd

In [26]:
def concat_update(old_path, new_df, columns=None, post_concat=None):
    """读取旧数据 -> concat新数据 -> 去重 -> [post_concat] -> columns reindex -> 写回"""
    old = pd.read_pickle(old_path)
    df = pd.concat([old, new_df], axis=0)
    df = df[~df.index.duplicated(keep='last')]
    if post_concat:
        df = post_concat(df)
    if columns is not None:
        df = df.reindex(columns=columns)
    df.to_pickle(old_path)
    return df

def pivot_update(raw_df, field, old_path, index='trade_date', col='code',
                 reindex_cols=None, post_pivot=None):
    """从长表 pivot -> concat_update"""
    new = raw_df.pivot(index=index, columns=col, values=field)
    if post_pivot:
        new = post_pivot(new)
    return concat_update(old_path, new, columns=reindex_cols)

# 更新交易日

In [27]:
update_start = '20260601'; update_end = '20260630'


In [28]:
# 校验：calendar 范围 + raw 数据日期完整性
new_cal = pd.read_pickle('./data_update/calendar.pkl')
new_cal_hk = pd.read_pickle('./data_update/calendar_hk.pkl')
start, end = pd.to_datetime(update_start, format='%Y%m%d'), pd.to_datetime(update_end, format='%Y%m%d')

for label, cal in [('calendar', new_cal), ('calendar_hk', new_cal_hk)]:
    dates = pd.to_datetime(cal.values, format='%Y%m%d')
    assert len(cal) > 0, f'{label} is empty!'
    assert (dates >= start).all() and (dates <= end).all(), \
        f'{label} dates outside [{update_start}, {update_end}]'
    print(f'{label} OK: {len(cal)} trading days in [{update_start}, {update_end}]')

def check_dates(dates_series, label, calendar=new_cal):
    missing = set(calendar.values) - set(dates_series.astype(str))
    assert len(missing) == 0, \
        f'[{label}] missing {len(missing)} calendar dates: {sorted(missing)[:10]}'
    print(f'[{label}] OK: {len(set(dates_series.astype(str)))} dates, covers all calendar')


calendar OK: 21 trading days in [20260601, 20260630]
calendar_hk OK: 21 trading days in [20260601, 20260630]


In [3]:
old_calendar = pd.read_pickle('./data/calendar.pkl')
new_calendar = pd.read_pickle('./data_update/calendar.pkl')
calendar = pd.concat([old_calendar, new_calendar]).drop_duplicates(keep='last').reset_index(drop=True)
calendar.to_pickle('./data/calendar.pkl')


# 更新上市天数

In [ ]:
new_listdays = pd.read_pickle('./data_update/listdays.pkl')
listdays = concat_update('./data/listdays.pkl', new_listdays,
                         post_concat=lambda x: x.fillna(0).astype(int))

# 更新st状态

In [ ]:
old_is_st = pd.read_pickle('./data/is_st.pkl')
is_st = pd.read_pickle('./data_update/is_st.pkl')
old_is_st = old_is_st.reindex(columns=listdays.columns, fill_value=False)
is_st = pd.concat([old_is_st, is_st], axis=0)
is_st = is_st[~is_st.index.duplicated(keep='last')]
is_st.to_pickle('./data/is_st.pkl')

code,000004.SZ,000005.SZ,600652.SH,600656.SH,600602.SH,600601.SH,600654.SH,600651.SH,600653.SH,000002.SZ,...,603293.SH,301513.SZ,920191.BJ,688820.SH,001312.SZ,688808.SH,920125.BJ,920156.BJ,920186.BJ,301599.SZ
20140102,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20140103,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20140106,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20140107,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20140108,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20260424,True,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20260427,True,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20260428,True,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20260429,True,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


# 更新日频行情

In [6]:
data = pd.read_pickle('./data_update/kline.pkl')
check_dates(data['trade_date'], 'kline')


In [7]:
data['_adjvwap'] = data['vwap'] * data['adjfactor']

fields = {
    'close': 'close', 'adjclose': 'adjclose', 'adjopen': 'adjopen',
    'adjhigh': 'adjhigh', 'adjlow': 'adjlow', '_adjvwap': 'adjvwap',
    'volume': 'volume', 'amount': 'amount', 'limit': 'limit', 'stopping': 'stopping',
}
for col, name in fields.items():
    pivot_update(data, col, f'./data/{name}.pkl', reindex_cols=listdays.columns)

# returns: percent -> decimal
pivot_update(data, 'returns', './data/returns.pkl',
             reindex_cols=listdays.columns, post_pivot=lambda x: x / 100)


# 更新总市值（换手率）

In [8]:
cap = pd.read_pickle('./data_update/cap.pkl')
check_dates(cap.index, 'cap')
cap = concat_update('./data/cap.pkl', cap, columns=listdays.columns)
cap


code,000004.SZ,000005.SZ,600652.SH,600656.SH,600602.SH,600601.SH,600654.SH,600651.SH,600653.SH,000002.SZ,...,603293.SH,301513.SZ,920191.BJ,688820.SH,001312.SZ,688808.SH,920125.BJ,920156.BJ,920186.BJ,301599.SZ
trade_dt,,,,,,,,,,,,,,,,,,,,,
20140102,9.917646e+05,2.276691e+06,2.267000e+06,1.143966e+06,5.743753e+06,6.079849e+06,5.587319e+06,4.323532e+06,4.365951e+06,9.027693e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20140103,9.909249e+05,2.249261e+06,2.267000e+06,1.157290e+06,5.513637e+06,5.970104e+06,5.466512e+06,4.220063e+06,4.313559e+06,8.859392e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20140106,1.032913e+06,2.176114e+06,2.267000e+06,1.147772e+06,5.239127e+06,5.750615e+06,5.277752e+06,4.146156e+06,4.121458e+06,8.498117e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20140107,1.027035e+06,2.240117e+06,2.267000e+06,1.151579e+06,5.267239e+06,5.750615e+06,5.187146e+06,4.138766e+06,4.086530e+06,8.432661e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20140108,1.027035e+06,2.203544e+06,2.267000e+06,1.107800e+06,5.200952e+06,5.816462e+06,5.194697e+06,4.257016e+06,4.034139e+06,8.448629e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20260424,3.852266e+05,NaN,NaN,NaN,2.532361e+07,4.923352e+07,1.079098e+07,1.657146e+07,3.347774e+06,4.205604e+07,...,9.264360e+06,8171000.0,2249373.672,1.747655e+08,6.167000e+06,8.203067e+07,2809123.2,NaN,NaN,NaN
20260427,3.653696e+05,NaN,NaN,NaN,2.574725e+07,5.107123e+07,1.087730e+07,1.672188e+07,3.484021e+06,4.177601e+07,...,9.305551e+06,8388000.0,2213894.592,1.883265e+08,5.812333e+06,9.034667e+07,3022531.2,4227272.0,NaN,NaN
20260428,3.653696e+05,NaN,NaN,NaN,2.462689e+07,4.957542e+07,1.061832e+07,1.629568e+07,3.445093e+06,4.194980e+07,...,8.809469e+06,7603000.0,2242277.856,1.732380e+08,5.467000e+06,8.880564e+07,2897856.0,4143976.0,NaN,NaN


# 更新ETF日频

In [9]:
etf_kline = pd.read_pickle('./data_update/etf_kline.pkl')
check_dates(etf_kline['trade_dt'], 'etf')


In [10]:
old_universe = pd.read_pickle('./data/etf_close.pkl').columns.tolist()
new_universe = sorted(list(set(etf_kline['s_info_windcode'].unique().tolist())-set(old_universe)))
universe = old_universe+new_universe

In [11]:
etf_fields = {
    'etf_close': 's_dq_adjclose', 'etf_open': 's_dq_adjopen',
    'etf_high': 's_dq_adjhigh', 'etf_low': 's_dq_adjlow',
    'etf_volume': 's_dq_volume', 'etf_amount': 's_dq_amount',
    'etf_trades': 'trades_count', 'etf_discount': 'discount_rate',
}
for name, field in etf_fields.items():
    pivot_update(etf_kline, field, f'./data/{name}.pkl',
                 index='trade_dt', col='s_info_windcode', reindex_cols=universe)

# returns: percent -> decimal
pivot_update(etf_kline, 's_dq_pctchange', './data/etf_returns.pkl',
             index='trade_dt', col='s_info_windcode', reindex_cols=universe,
             post_pivot=lambda x: x / 100)


# 更新资金流

In [12]:
mf = pd.read_pickle('./data_update/mf.pkl')
check_dates(mf['trade_dt'], 'mf')


In [13]:
mf_fields = {
    'ord': 'trades_count', 'ordM1': 'buy_trades_med_order', 'ordM2': 'sell_trades_med_order',
    'ordS1': 'buy_trades_small_order', 'ordS2': 'sell_trades_small_order',
    'volM1': 'buy_volume_med_order', 'volM2': 'sell_volume_med_order',
    'volS1': 'buy_volume_small_order', 'volS2': 'sell_volume_small_order',
    'ordL1': 'buy_trades_large_order', 'ordL2': 'sell_trades_large_order',
    'volL1': 'buy_volume_large_order', 'volL2': 'sell_volume_large_order',
    'volNetOpen': 's_mfd_inflow_openvolume', 'volNetClose': 's_mfd_inflow_closevolume',
    'volS1a': 'buy_volume_small_order_act', 'volS2a': 'sell_volume_small_order_act',
    'volM1a': 'buy_volume_med_order_act', 'volM2a': 'sell_volume_med_order_act',
    'volL1a': 'buy_volume_large_order_act', 'volL2a': 'sell_volume_large_order_act',
}
for name, field in mf_fields.items():
    pivot_update(mf, field, f'./data/{name}.pkl', index='trade_dt', col='s_info_windcode', reindex_cols=listdays.columns)


# 更新停牌

In [ ]:
is_suspend = pd.read_pickle('./data/amount.pkl')==0
is_suspend.to_pickle('./data/is_suspend.pkl')

# 更新行业分类

In [ ]:
indint = pd.read_pickle('./data_update/indint.pkl')
check_dates(indint.index, 'indint')
indint = concat_update('./data/indint.pkl', indint, columns=listdays.columns,
                       post_concat=lambda x: x.fillna(-1).astype(int))
indint

# 更新barra

In [17]:
# factors = pd.read_pickle('./data_update/factors.pkl')

# liq = factors.pivot(index='trade_date', columns='code', values='liq')
# old_liq = pd.read_pickle('./data/liq.pkl')
# liq = pd.concat([old_liq, liq], axis=0)
# liq = liq.reindex(columns=listdays.columns)
# liq.sort_index().to_pickle('./data/liq.pkl')

# nlsize = factors.pivot(index='trade_date', columns='code', values='nlsize')
# old_nlsize = pd.read_pickle('./data/nlsize.pkl')
# nlsize = pd.concat([old_nlsize, nlsize], axis=0)
# nlsize = nlsize.reindex(columns=listdays.columns)
# nlsize.sort_index().to_pickle('./data/nlsize.pkl')

# size = factors.pivot(index='trade_date', columns='code', values='size')
# old_size = pd.read_pickle('./data/size.pkl')
# size = pd.concat([old_size, size], axis=0)
# size = size.reindex(columns=listdays.columns)
# size.sort_index().to_pickle('./data/size.pkl')

# beta = factors.pivot(index='trade_date', columns='code', values='beta')
# old_beta = pd.read_pickle('./data/beta.pkl')
# beta = pd.concat([old_beta, beta], axis=0)
# beta = beta.reindex(columns=listdays.columns)
# beta.sort_index().to_pickle('./data/beta.pkl')

# resvol = factors.pivot(index='trade_date', columns='code', values='resvol')
# old_resvol = pd.read_pickle('./data/resvol.pkl')
# resvol = pd.concat([old_resvol, resvol], axis=0)
# resvol = resvol.reindex(columns=listdays.columns)
# resvol.sort_index().to_pickle('./data/resvol.pkl')

# mom = factors.pivot(index='trade_date', columns='code', values='mom')
# old_mom = pd.read_pickle('./data/mom.pkl')
# mom = pd.concat([old_mom, mom], axis=0)
# mom = mom.reindex(columns=listdays.columns)
# mom.sort_index().to_pickle('./data/mom.pkl')

# # 20日反转
# returns = pd.read_pickle('./data/returns.pkl')
# ret20 = returns.rolling(20).sum()
# ret20.to_pickle('./data/ret20.pkl')

# 更新市场指数

In [ ]:
mkt = pd.read_pickle('./data_update/mkt.pkl')
check_dates(mkt['trade_date'], 'mkt')
index_list = {'zzqz': '000985.CSI', 'sz50': '000016.SH', 'hs300': '000300.SH', 'zz500': '000905.SH', 'zz1000': '000852.SH'}
for inst in index_list.keys():
    df = mkt[mkt['code']==index_list[inst]].sort_values('trade_date').set_index('trade_date')
    concat_update(f'./data/{inst}.pkl', df)

# 更新港股

In [19]:
old_calendar_hk = pd.read_pickle('./data/calendar_hk.pkl')
new_calendar_hk = pd.read_pickle('./data_update/calendar_hk.pkl')
calendar_hk = pd.concat([old_calendar_hk, new_calendar_hk]).drop_duplicates(keep='last').reset_index(drop=True)
calendar_hk.to_pickle('./data/calendar_hk.pkl')


In [20]:
index_list = {'hsi': 'HSI.HI', 'hstech': 'HSTECH.HI'}
hkmkt = pd.read_pickle('./data_update/hkmkt.pkl')
check_dates(hkmkt['trade_date'], 'hkmkt', calendar=new_cal_hk)
for inst in index_list.keys():
    hk = hkmkt[hkmkt['code']==index_list.get(inst)]
    old_hk = pd.read_pickle(f'./data/{inst}.pkl')
    hk = pd.concat([old_hk, hk], axis=0)
    hk = hk.drop_duplicates(keep='last')
    hk.to_pickle(f'./data/{inst}.pkl')


# 更新指数成份信息

In [ ]:
pd.set_option('future.no_silent_downcasting', True)
codes = {'zz1000': '000852.SH', 'zzqz': '000985.CSI', 'zz500': '000905.SH', 'hs300': '000300.SH', 'zz2000': '932000.CSI'}
keys = list(codes.keys()); values = list(codes.values())
mb_info = pd.read_pickle('./data_update/mb_info.pkl')
for key in keys:
    new_is_mb = pd.DataFrame(False, index=new_calendar.values, columns=listdays.columns)
    mb_info2 = mb_info.query('s_info_windcode==@codes[@key]', engine='python')
    for name, row in mb_info2.iterrows():
        new_is_mb.loc[(new_is_mb.index>=row['s_con_indate'])&(new_is_mb.index<=row['s_con_outdate']), row['s_con_windcode']] = True
    concat_update(f'./data/mb_{key}.pkl', new_is_mb,
                  post_concat=lambda x: x.fillna(False).astype(bool))